# Tutorial 6: Model Interpretability

**Big Data in Finance** | Dr Daniele Bianchi  
**Duration:** 1 hour

---

## Learning Objectives

By the end of this tutorial, you will be able to:

1. Train a `GradientBoostingClassifier` for credit default prediction
2. Evaluate it with AUC and a classification report
3. Explain the model **globally** with SHAP (which features matter, and how)
4. Explain **individual** predictions with SHAP waterfall plots (and, optionally, LIME)
5. Read **partial dependence plots** to understand average feature effects

---

## Context: Why Interpretability?

Black-box models are not enough in finance. Regulators (e.g. SR 11-7, ECOA) require lenders to explain credit decisions, and risk managers need to trust *why* a model makes a prediction. We first train a default-prediction model, then spend most of the tutorial **interpreting** it.

---

## Part 1: Setup and Model Training

First, we load the data, preprocess it, and train a **GradientBoostingClassifier** to predict credit default.

> ⚠️ **Leakage caveat:** This LendingClub-style dataset includes `int_rate` and `grade`, which are LendingClub's *own* risk-based price and rating. Because they already encode LendingClub's default expectation, using them as features makes the task easier than a genuine origination-time model would be. We keep them here for simplicity, but in practice you would drop them (or model them separately) to avoid this subtle leakage.

> 💡 **Note (scope):** This tutorial uses a deliberately simplified setup for a one-hour session, so its numbers will not exactly reproduce the full results in the lecture slides and notes.

In [ ]:
# =============================================================================
# Import libraries
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report, RocCurveDisplay

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries loaded successfully!")
print("Model class: GradientBoostingClassifier")

In [ ]:
# =============================================================================
# Load LendingClub tutorial dataset (provided CSV)
# =============================================================================

DATA_PATH = "lending_club_tutorial.csv"  # adjust if the file is in a different folder

df = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH}")
print(f"Dataset shape: {df.shape}")

# Basic sanity checks
assert 'default' in df.columns, "Expected target column 'default'"
df['default'] = df['default'].astype(int)      # ensure target is 0/1 int
df.columns = [c.strip() for c in df.columns]    # tidy column names

# Missingness overview
missing = df.isna().mean().sort_values(ascending=False)
print("\nTop missingness rates:")
print((100 * missing.head(10)).round(2).astype(str) + "%")

# Identify numeric vs categorical columns (excluding the target)
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != 'default']
cat_cols = [c for c in df.columns if c not in num_cols and c != 'default']

# Categorical missing values -> explicit 'Missing' label.
# (A constant, not a data-derived statistic, so it is safe to apply before the split.)
for c in cat_cols:
    if df[c].isna().any():
        df[c] = df[c].fillna('Missing')

# One-hot encode categoricals (keeps the notebook self-contained)
df_model = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print("\nModel matrix shape after one-hot encoding:", df_model.shape)
print("Default rate:", f"{df_model['default'].mean()*100:.1f}%")

# NOTE: numeric missing values are imputed AFTER the train/test split,
# using training-set medians only, to avoid data leakage (see next cell).

In [ ]:
# =============================================================================
# Prepare data and train/test split
# =============================================================================

# Features and target
X = df_model.drop(columns=['default'])
y = df_model['default']

# Keep feature names for interpretation plots
feature_names = X.columns.tolist()

# Train/test split (stratified to preserve the default rate)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Impute remaining numeric missing values using TRAINING medians only (no leakage)
train_medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Number of features: {X_train.shape[1]}")
print(f"Default rate (train): {y_train.mean()*100:.2f}%")
print(f"Default rate (test):  {y_test.mean()*100:.2f}%")
print(f"Remaining missing values: train={int(X_train.isna().sum().sum())}, test={int(X_test.isna().sum().sum())}")

# We keep X_train/X_test as pandas DataFrames (with column names) for SHAP/PDP.

In [ ]:
# =============================================================================
# Train Gradient Boosting model
# =============================================================================

model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

model.fit(X_train, y_train)

# Evaluate
y_pred_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)

print(f"Test AUC: {auc:.3f}")
print("\nModel trained successfully!")

---

## Part 2: Evaluating the Model

Before interpreting the model, we generate predicted default probabilities on the test set and evaluate performance.

In [ ]:
# =============================================================================
# Evaluate the model
# =============================================================================

# Probabilities and hard predictions
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_pred_proba)
print(f"Test AUC: {auc:.3f}")
print("\nClassification report (threshold = 0.5):")
print(classification_report(y_test, y_pred))

# ROC curve
RocCurveDisplay.from_predictions(y_test, y_pred_proba)
plt.title("ROC curve (test set)")
plt.show()

# Show a few example predictions
pred_table = (
    pd.DataFrame({
        "y_true": y_test.values,
        "p_default": y_pred_proba,
    })
    .assign(pred_rank=lambda d: d["p_default"].rank(ascending=False, method="first"))
    .sort_values("p_default", ascending=False)
)

print("\nTop 5 highest predicted risk cases (test set):")
print(pred_table.head(5).to_string(index=False))

print("\nTop 5 lowest predicted risk cases (test set):")
print(pred_table.tail(5).to_string(index=False))

---

## Part 3: Model Interpretability

A credit model is only useful if we can **explain** its predictions — to risk managers, to regulators, and to applicants who are turned down. In this part we open up the trained `GradientBoostingClassifier` using three complementary tools:

- **SHAP** (SHapley Additive exPlanations): which features drive predictions, in which direction, both overall and for an individual loan.
- **Partial Dependence Plots (PDP)**: the average effect of a feature on the predicted default probability.
- **LIME** (optional): a local linear approximation that explains a single prediction.

We reuse the `model`, `X_train`, and `X_test` from Part 1 — no retraining needed.

### 3.1 Global importance with SHAP

The **bar** plot ranks features by their average impact; the **beeswarm** plot adds direction (red = high feature value) and the spread of effects across loans.

In [ ]:
# =============================================================================
# SHAP: global feature importance
# =============================================================================
import shap

# TreeExplainer is exact and fast for tree-based models like GradientBoosting
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Some SHAP versions return a list [class 0, class 1] for binary classifiers;
# keep the contributions for the positive ("default") class.
if isinstance(shap_values, list):
    shap_values = shap_values[1]

# Mean absolute SHAP value = average impact of each feature on the prediction
importance = (pd.Series(np.abs(shap_values).mean(axis=0), index=X_test.columns)
                .sort_values(ascending=False))
print("Top 10 features by mean |SHAP|:")
print(importance.head(10).round(4))

# Bar summary (global importance) and beeswarm (direction + magnitude)
shap.summary_plot(shap_values, X_test, plot_type="bar", show=True)
shap.summary_plot(shap_values, X_test, show=True)

### 3.2 Local explanations: one feature, one loan

The **dependence plot** shows how the SHAP contribution of the most important feature varies with its value (and reveals interactions through the colour). The **waterfall plot** decomposes a *single* prediction: starting from the average prediction (the base value), each feature pushes the predicted log-odds of default up or down.

In [ ]:
# =============================================================================
# SHAP: how one feature acts, and explaining a single loan
# =============================================================================

# Most important feature (by mean |SHAP|)
top_feature = importance.index[0]

# Dependence plot: SHAP value of the top feature vs its value (colour shows interaction)
shap.dependence_plot(top_feature, shap_values, X_test, show=True)

# Explain the single highest-risk test loan with a waterfall plot
idx = int(np.argmax(y_pred_proba))                         # most likely to default
base_value = np.atleast_1d(explainer.expected_value)[-1]   # positive-class base (log-odds)

explanation = shap.Explanation(
    values=shap_values[idx],
    base_values=base_value,
    data=X_test.iloc[idx].values,
    feature_names=X_test.columns.tolist(),
)
shap.plots.waterfall(explanation, max_display=12, show=True)

print(f"Explained loan #{idx}: predicted default probability = {y_pred_proba[idx]:.2%}, "
      f"actual outcome = {'default' if y_test.iloc[idx] == 1 else 'repaid'}")

### 3.3 Partial Dependence Plots

SHAP attributes a contribution to each observation. A **partial dependence plot** instead shows the *average* model prediction as one feature is varied across its range, holding the others fixed. It answers: "all else equal, how does default risk change with this feature?"

In [ ]:
# =============================================================================
# Partial Dependence Plots (PDP)
# =============================================================================
from sklearn.inspection import PartialDependenceDisplay

# PDPs for the two most important features (by mean |SHAP|)
pdp_features = list(importance.index[:2])

fig, ax = plt.subplots(figsize=(11, 4))
PartialDependenceDisplay.from_estimator(model, X_train, features=pdp_features, ax=ax)
fig.suptitle("Partial dependence: average effect on predicted default")
plt.tight_layout()
plt.show()

### 3.4 LIME (optional)

**LIME** (Local Interpretable Model-agnostic Explanations) explains a single prediction by fitting a simple, interpretable model in the local neighbourhood of that point. It is model-agnostic and complements SHAP. This cell is optional and requires the `lime` package (`pip install lime`).

In [ ]:
# =============================================================================
# LIME (optional) -- requires:  pip install lime
# =============================================================================
try:
    from lime.lime_tabular import LimeTabularExplainer

    lime_explainer = LimeTabularExplainer(
        training_data=X_train.values,
        feature_names=feature_names,
        class_names=["Repaid", "Default"],
        mode="classification",
    )

    # Explain the same high-risk loan we examined with SHAP
    lime_exp = lime_explainer.explain_instance(
        data_row=X_test.iloc[idx].values,
        predict_fn=model.predict_proba,
        num_features=10,
    )
    lime_exp.as_pyplot_figure()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("LIME is not installed. Run `pip install lime` to try this cell.")

---

## Wrap-up

You now have an end-to-end, **interpretable** credit-risk workflow:
- load and preprocess a real credit dataset (imputing on the training set only)
- train a `GradientBoostingClassifier` and evaluate it with AUC / classification report
- explain it **globally** (SHAP summary, PDP) and **locally** (SHAP waterfall, optional LIME)

Optional next steps (not covered here): hyperparameter tuning, cross-validation, probability calibration, and threshold selection based on business costs.